# A/B Reranker trên Colab T4 (free) — Zalo LTR 2021

So `AITeamVN/Vietnamese_Reranker` (self-host, ứng viên "vượt trần") với `qwen3-rerank` (production API) trên benchmark công khai Zalo LTR (MIT).

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

Chạy lần lượt từng cell. Cần: 1 GitHub PAT (repo private) + QWEN_API_KEY (chỉ cho arm production).

In [ ]:
# 1) Xác nhận có GPU T4. Nếu báo 'not connected to GPU' → Runtime > Change runtime type > T4 GPU
!nvidia-smi

In [ ]:
# 2) Clone repo private (dev) — dán GitHub PAT (Settings > Developer settings > Fine-grained token, quyền Contents: Read)
import os
GITHUB_TOKEN = ""  # <-- DÁN PAT vào đây
!git clone https://{GITHUB_TOKEN}@github.com/trungnguyen1618033/legal-guard-PH.git
os.chdir('/content/legal-guard-PH')
!git checkout dev && git log --oneline -1
print('CWD:', os.getcwd())

In [ ]:
# 3) Cài deps TỐI THIỂU vào python hệ thống của Colab (KHÔNG dùng uv → giữ torch-CUDA sẵn có).
#    sentence-transformers dùng luôn torch+CUDA của Colab. Còn lại chỉ để import qwen adapter.
!pip install -q sentence-transformers pydantic pydantic-settings httpx huggingface_hub

In [ ]:
# 4) SMOKE test 40 query trước (nhanh ~1 phút) — kiểm mọi thứ chạy được đã, rồi mới chạy full.
!python -m evaluation.rerank_ab --reranker hf:AITeamVN/Vietnamese_Reranker --limit 40

In [ ]:
# 5) ARM self-host FULL (788 query) — chạy trên GPU T4. Lưu report riêng để không bị arm sau đè.
!python -m evaluation.rerank_ab --reranker hf:AITeamVN/Vietnamese_Reranker
!cp evaluation/rerank_ab_report.json evaluation/rerank_ab_AITeamVN.json
print('Đã lưu evaluation/rerank_ab_AITeamVN.json')

In [ ]:
# 6) ARM production qwen3-rerank (API, cần key) — head-to-head trên CÙNG 788 query.
import os
os.environ['QWEN_API_KEY'] = ''  # <-- DÁN QWEN key
!python -m evaluation.rerank_ab --reranker qwen3-api
!cp evaluation/rerank_ab_report.json evaluation/rerank_ab_qwen.json
print('Đã lưu evaluation/rerank_ab_qwen.json')

In [ ]:
# 7) SO KẾT QUẢ + verdict
import json
a = json.load(open('evaluation/rerank_ab_AITeamVN.json'))
q = json.load(open('evaluation/rerank_ab_qwen.json'))
base = a['bm25_baseline']['mrr@10']
am, qm = a['reranked']['mrr@10'], q['reranked']['mrr@10']
print(f"BM25 baseline        MRR@10 = {base:.4f}")
print(f"AITeamVN (self-host) MRR@10 = {am:.4f}  (lift {am-base:+.4f})  Recall@10={a['reranked']['recall@10']:.1%}")
print(f"qwen3-rerank (prod)  MRR@10 = {qm:.4f}  (lift {qm-base:+.4f})  Recall@10={q['reranked']['recall@10']:.1%}")
print()
diff = am - qm
if diff > 0.02:
    print(f"=> AITeamVN THẮNG rõ (+{diff:.4f}). Cân nhắc self-host (TEI) NẾU tải rerank cao.")
elif diff < -0.02:
    print(f"=> qwen3-rerank vẫn hơn ({diff:.4f}). Giữ API. Đóng hồ sơ A/B.")
else:
    print(f"=> Ngang nhau ({diff:+.4f}). Giữ API (rẻ 10-100x, zero-ops). Đóng hồ sơ A/B.")

In [ ]:
# 8) Tải 2 report về máy để lưu vào repo (git)
from google.colab import files
files.download('evaluation/rerank_ab_AITeamVN.json')
files.download('evaluation/rerank_ab_qwen.json')